# Autobot Gemma 4 Developer Agent · Experiment 9 SOTA
## Autonomous Software Engineering Agent for the Gemma 4 Developer Agent Challenge ($65,000 USD)
### Architecture: Dynamic Problem Triage + First-Turn AST Grounding + Ephemeral `/tmp` Repro + Resilient Context Anchoring

**Key Upgrades in Experiment 9**:
1. **Dynamic Problem Triage & Fast-Exit**: Turn-1 complexity classifier. Bails in 15 seconds on intractable multi-package refactors; preserves deep 4.5-minute budgets for high-probability localized defects.
2. **First-Turn AST Graph Tool Protocol**: Targets `search_similar_code(symbol)`, `get_code_neighbors(node)`, and `get_code_subgraph(nodes)`. Replaces blind 10-call grep loops with 1-call exact symbol resolution.
3. **Ephemeral `/tmp/repro.py` Verification**: Isolated reproduction script in `/tmp/` guarantees clean `/workspace` git diffs while proving bug reproduction and fix validity.
4. **Resilient 4-to-6 Line Context Anchoring**: Eliminates `FileEditError: Multiple occurrences found` with wide anchor windows and whole-function replacement fallback.
5. **Strict Test Anti-Tampering Immunity**: Guarantees zero edits to `tests/`, `conftest.py`, or `pytest.ini`, preventing Container B automatic test rollback failures.


In [ ]:
import os
import sys
import shutil
import zipfile
import hashlib
import json
from pathlib import Path

WORKING_DIR = Path('/kaggle/working').resolve()
AGENT_DIR = WORKING_DIR / 'submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)

AGENT_DIR.mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'configs').mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'prompts').mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'sub_agents').mkdir(parents=True, exist_ok=True)

print(f"[INIT] Initialized submission build directory at: {AGENT_DIR}")


In [ ]:
# 1. Write configs/sampling.yaml
SAMPLING_YAML = "temperature: 0.15\ntop_p: 0.92\ntop_k: 40\nmax_output_tokens: 16384\nseed: 42\nthinking_config:\n  thinking_budget: 3072\n  include_thoughts: true\n"
(AGENT_DIR / 'configs' / 'sampling.yaml').write_text(SAMPLING_YAML.strip() + '\n', encoding='utf-8')

# 2. Write configs/eval_config.yaml
EVAL_YAML = "evaluation:\n  timeout_seconds: 90\n  max_tool_calls: 25\n  max_time_minutes: 4.5\n  max_turns: 35\n"
(AGENT_DIR / 'eval_config.yaml').write_text(EVAL_YAML.strip() + '\n', encoding='utf-8')

# 3. Write prompts/analyzer.md
ANALYZER_MD = "You are `code_analyzer`, a read-only code navigation and root-cause localization specialist in `/workspace`.\nYou never modify files or run tests. Your single purpose is to identify the exact file, line range, and defect mechanism for the reported issue in the fewest possible tool calls.\n\n## Available Code Intelligence Tools\n- `search_similar_code(query)`: Query semantic code embeddings using a single exact symbol name (e.g., `HTTPConnection`, `parse_header`, `ChunkedEncodingError`), NOT natural language.\n- `get_code_neighbors(node)`: Inspect callers, callees, and dependencies around a known symbol or function.\n- `get_code_subgraph(nodes)`: View structural connections between candidate symbols.\n- `read_file(filepath, start_line, end_line)`: Inspect code with tight line windows (<= 150 lines).\n- `run_command(command)`: Use strictly for fast, read-only discovery (`rg -n \"<symbol>\"`, `find . -name \"*.py\"`).\n\n## Fast Localization Protocol\n1. **Identifier Extraction**:\n   - Extract exact symbol names, class names, function names, parameter names, or exception types from the issue description.\n2. **First-Action Graph Search**:\n   - Call `search_similar_code(\"<exact_symbol>\")` or `rg -n \"def <function_name>\"` immediately.\n   - Do NOT run generic find commands across the entire repo.\n3. **Trace Call Chain**:\n   - Use `get_code_neighbors(\"<symbol>\")` or follow caller/callee logic to locate the defect point.\n4. **Code Inspection**:\n   - Read the target function with `read_file`. Confirm exact line numbers and baseline logic.\n5. **Identify Target Test**:\n   - Note the relevant test file under `tests/` that exercises this module.\n\n## Output Format\nConclude with a structured Bug Dossier (at most 200 words):\n- LOCATION: `<path/to/file.py>:<start_line>-<end_line>` (`<class_or_function>`)\n- ROOT CAUSE: `<concise 1-2 sentence explanation of the defect>`\n- DEFECT TYPE: `missing_none_check | boundary_condition | off_by_one | parameter_mismatch | error_handling`\n- SUGGESTED FIX: `<concrete minimal modification required in library implementation>`\n- TARGET TEST: `<tests/path/to/test_file.py::test_name>`\n- TRIAGE: `HIGH_SOLVABILITY` (single-file localized bug) or `LOW_SOLVABILITY` (massive multi-file architectural rewrite)\n"
(AGENT_DIR / 'prompts' / 'analyzer.md').write_text(ANALYZER_MD.strip() + '\n', encoding='utf-8')

# 4. Write prompts/system.md
SYSTEM_MD = "You are Autobot SWE-Coder, an elite autonomous software engineering agent in `/workspace`.\nYour goal is to solve the reported repository issue with a clean, minimal, working patch that passes hidden tests in a fresh verification environment.\nAlways conclude your session by calling `submit_patch()`.\n\n## Core Directives & Hard Constraints\n\n1. **Strict Test Anti-Tampering Immunity**:\n   - The evaluation sandbox (`Container B`) forcefully wipes any changes made to test files (`test_*.py`, `*_test.py`, `tests/`), runner configurations (`pytest.ini`, `conftest.py`, `setup.cfg`, `pyproject.toml`), or package stubs (`git checkout HEAD -- <test_files>`).\n   - **NEVER modify or delete test files to make tests pass.**\n   - All code modifications MUST be strictly within the library/package implementation source code.\n\n2. **Workflow Phases**:\n\n   - **Phase 1: Scout, Triage & Fast-Exit**:\n     - Delegate initial localization to `code_analyzer` via your `agent_tool`.\n     - Review the Bug Dossier:\n       - **Fast-Exit Guard**: If the diagnosis indicates `LOW_SOLVABILITY` (e.g., massive multi-package refactoring, complex C-extensions, or architectural deprecation requiring $>100$ lines across $\\ge 5$ files), immediately call `submit_patch()` and exit cleanly to conserve compute budget for solvable problems.\n       - If `HIGH_SOLVABILITY` (localized bug in 1\u20132 files), proceed immediately to Phase 2.\n\n   - **Phase 2: Ephemeral Reproduction (`/tmp/repro.py`)**:\n     - Create a minimal 5-line reproducing script in **`/tmp/repro.py`** (NOT in `/workspace`):\n       ```bash\n       run_command(\"cat << 'EOF' > /tmp/repro.py\\nimport <module>\\n# minimal trigger\\nEOF\\npython3 /tmp/repro.py\")\n       ```\n     - Observe the expected error or traceback on the unpatched codebase.\n     - Keeping reproduction scripts in `/tmp/` ensures `/workspace` stays completely clean and prevents scratch files from polluting `git diff`.\n\n   - **Phase 3: Resilient Surgical Implementation (`edit_file`)**:\n     - Modify the exact target function in the source code.\n     - **Anchor Rules**: Always include 4 to 6 lines of unchanged surrounding code before and after the change in `old_string` to guarantee exact, unique matching.\n     - Copy `old_string` character-for-character from `read_file`.\n     - **Fallback Rule**: If `edit_file` returns `FileEditError: Multiple occurrences found`, do NOT repeat the same call. Instead, widen the context window to include the entire enclosing function definition (`def func(...)` down to `return`).\n\n   - **Phase 4: Hermetic Verification**:\n     - 1. Verify syntax: `run_command(\"python3 -m py_compile <path/to/modified_file.py>\")`.\n     - 2. Rerun the reproduction script: `run_command(\"python3 /tmp/repro.py\")`. Confirm exit code 0 and successful output.\n     - 3. If a targeted test was identified, run ONLY that test: `run_command(\"pytest <test_file.py> -k '<test_name>' -v --tb=short\")`.\n     - NEVER run the full repository test suite (this triggers timeouts).\n\n   - **Phase 5: Diff Audit & Submission**:\n     - Inspect changes: `run_command(\"git status --short\")` and `run_command(\"git diff --stat\")`.\n     - Verify no trailing syntax warnings: `run_command(\"git diff --check\")`.\n     - Clean up any temporary files in `/tmp`.\n     - **Call `submit_patch()`** as your final tool action. It captures all changes into the official evaluation artifact.\n     - Provide a concise 2-sentence summary of the fix and verification outcome.\n\n3. **Tool & Budget Discipline**:\n   - Aim for 3\u20135 calls exploring, 2\u20133 calls editing, and 2\u20133 calls verifying.\n   - The environment is completely offline. Pre-installed wheels exist under `/wheels`; never run `pip install` or network commands.\n   - `submit_patch()` and `get_status()` are free tool actions that do not consume tool call limits.\n"
(AGENT_DIR / 'prompts' / 'system.md').write_text(SYSTEM_MD.strip() + '\n', encoding='utf-8')

# 5. Write sub_agents/code_analyzer.yaml
CODE_ANALYZER_YAML = "name: code_analyzer\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Read-only code navigation and root-cause localization specialist. Given an issue, returns exact file location, root cause, and target test.\ninstruction: !include ../prompts/analyzer.md\ngenerate_content_config: !include ../configs/sampling.yaml\ntools:\n  - run_command\n  - read_file\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\n"
(AGENT_DIR / 'sub_agents' / 'code_analyzer.yaml').write_text(CODE_ANALYZER_YAML.strip() + '\n', encoding='utf-8')

# 6. Write root agent.yaml
AGENT_YAML = "name: autobot_swe_coder\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Autonomous software engineer that resolves repository issues with surgical, verified patches.\ninstruction: !include prompts/system.md\ngenerate_content_config: !include configs/sampling.yaml\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - agent_tool:\n      config_path: sub_agents/code_analyzer.yaml\n      skip_summarization: true\n"
(AGENT_DIR / 'agent.yaml').write_text(AGENT_YAML.strip() + '\n', encoding='utf-8')

print("[SUCCESS] Wrote all 6 declarative configuration and prompt files to submission bundle.")


In [ ]:
# Package submission.zip with deterministic timestamps and POSIX file attributes
ZIP_PATH = WORKING_DIR / 'submission.zip'
ZIP_PATH.unlink(missing_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(p for p in AGENT_DIR.rglob('*') if p.is_file()):
        rel_path = str(p.relative_to(AGENT_DIR)).replace('\\', '/')
        info = zipfile.ZipInfo(rel_path, date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o644 << 16
        zf.writestr(info, p.read_bytes())

size_kb = ZIP_PATH.stat().st_size / 1024.0
print(f"[PACKAGE] Successfully generated {ZIP_PATH.name} ({size_kb:.2f} KB)")
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    for name in zf.namelist():
        print(f"  - {name}")


In [ ]:
import yaml

# Safe custom YAML loader that handles Google ADK !include tags
class SafeIncludeLoader(yaml.SafeLoader):
    pass

def include_constructor(loader, node):
    return f"!include {loader.construct_scalar(node)}"

SafeIncludeLoader.add_constructor('!include', include_constructor)

print("=== AUTOBOT SWE-GEMMA SUBMISSION VALIDATION SUITE ===")

# Gate 1: Archive Size Check (< 3 GiB limit)
archive_bytes = ZIP_PATH.stat().st_size
assert archive_bytes < 3 * 1024 * 1024 * 1024, f"Archive exceeds 3 GiB: {archive_bytes} bytes"
print(f"Gate 1 Passed: Archive size {archive_bytes / 1024:.2f} KB is well within 3 GiB ceiling.")

# Gate 2: Required Root Config Discovery
root_agent_path = AGENT_DIR / 'agent.yaml'
assert root_agent_path.is_file(), "Missing required root agent.yaml"
print("Gate 2 Passed: Root agent.yaml discovered at submission root.")

# Gate 3: Parse and Validate Root Agent YAML
with open(root_agent_path, 'r', encoding='utf-8') as f:
    root_agent_data = yaml.load(f, Loader=SafeIncludeLoader)

assert root_agent_data.get('name') == 'autobot_swe_coder', f"Unexpected agent name: {root_agent_data.get('name')}"
print(f"Gate 3 Passed: Root agent verified: {root_agent_data.get('name')}")

# Gate 4: Single Base Model Rule (Hard Competition Constraint)
ALLOWED_MODEL = 'gemma-4-31b-it-qat-w4a16-ct'
assert root_agent_data.get('model') == ALLOWED_MODEL, f"Invalid root model: {root_agent_data.get('model')}"

analyzer_path = AGENT_DIR / 'sub_agents' / 'code_analyzer.yaml'
assert analyzer_path.is_file(), "Missing sub_agents/code_analyzer.yaml"
with open(analyzer_path, 'r', encoding='utf-8') as f:
    analyzer_data = yaml.load(f, Loader=SafeIncludeLoader)
assert analyzer_data.get('model') == ALLOWED_MODEL, f"Invalid analyzer model: {analyzer_data.get('model')}"
print(f"Gate 4 Passed: Single Base Model verified strictly as '{ALLOWED_MODEL}' across all agents.")

# Gate 5: Tool Binding & Declarative Schema Check
root_tools = root_agent_data.get('tools', [])
assert 'submit_patch' in root_tools, "Root agent missing mandatory submit_patch tool!"
assert 'edit_file' in root_tools, "Root agent missing edit_file tool!"
assert 'read_file' in root_tools, "Root agent missing read_file tool!"
assert 'run_command' in root_tools, "Root agent missing run_command tool!"
print(f"Gate 5 Passed: All required workspace tools declared ({len(root_tools)} tools bound).")

# Gate 6: Operational Budget Limits (eval_config.yaml)
eval_cfg_path = AGENT_DIR / 'eval_config.yaml'
assert eval_cfg_path.is_file(), "Missing eval_config.yaml"
with open(eval_cfg_path, 'r', encoding='utf-8') as f:
    eval_cfg = yaml.safe_load(f).get('evaluation', {})

assert eval_cfg.get('max_time_minutes', 0) <= 60.0, f"Exceeds max time minutes: {eval_cfg.get('max_time_minutes')}"
assert eval_cfg.get('max_tool_calls', 0) <= 100, f"Exceeds max tool calls: {eval_cfg.get('max_tool_calls')}"
assert eval_cfg.get('timeout_seconds', 0) <= 300, f"Exceeds command timeout: {eval_cfg.get('timeout_seconds')}"
print(f"Gate 6 Passed: Operational budgets verified: {eval_cfg['max_time_minutes']}m / {eval_cfg['max_tool_calls']} tools / {eval_cfg['timeout_seconds']}s timeout.")

# Gate 7: Sampling & Thinking Configuration (configs/sampling.yaml)
sampling_path = AGENT_DIR / 'configs' / 'sampling.yaml'
assert sampling_path.is_file(), "Missing configs/sampling.yaml"
with open(sampling_path, 'r', encoding='utf-8') as f:
    sampling_data = yaml.safe_load(f)

thinking_cfg = sampling_data.get('thinking_config', {})
assert thinking_cfg.get('include_thoughts') is True, "Thinking must be enabled for SWE-bench reasoning!"
assert 0 < thinking_cfg.get('thinking_budget', 0) <= 32768, f"Invalid thinking budget: {thinking_cfg.get('thinking_budget')}"
print(f"Gate 7 Passed: Gemma 4 thinking engine verified: budget={thinking_cfg['thinking_budget']} tokens.")

# Gate 8: Checksum and Artifact Audit
sha256_hash = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print(f"Gate 8 Passed: submission.zip SHA256: {sha256_hash}")

print("\n" + "=" * 65)
print("ALL 8 ADK COMPETITION VALIDATION GATES PASSED CLEANLY (100% READY FOR SUBMISSION)")
print("=" * 65)
